In [47]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn. metrics import r2_score 
import pickle
from sklearn.metrics import mean_squared_error
from scipy import stats

In [61]:
df = pd.read_csv("cleaned_data.csv", low_memory=False)

In [62]:
for i in df:
    if i != 'stm_progfh_t_fh':
        df[i] = (df[i] - df[i].mean()) / df[i].std()
df.head(20)

,stm_sap_meld_ddt,stm_geo_mld,stm_equipm_nr_mld,stm_prioriteit,stm_aanngeb_dd,stm_oorz_groep,stm_oorz_code,stm_contractgeb_gst,stm_techn_gst,stm_progfh_in_duur,stm_fh_status,stm_progfh_t_fh
0,-1.618463,0.713558,0.416338,0.279343,-1.618114,0.568630,0.776524,-0.100931,0.995238,-0.405484,-0.502752,55.0
1,-1.618377,0.857239,-0.420474,-1.113964,-1.618114,-1.634060,-1.346253,-0.701590,-0.156969,-0.584482,-0.502752,13.0
2,-1.618360,0.857239,NaN,-1.113964,-1.618114,0.568630,1.274460,-0.701590,-1.309176,-0.512883,-0.502752,44.0
3,-1.618342,-1.063180,-0.734000,-1.113964,-1.618114,-1.634060,-1.215218,0.559794,0.867215,-0.918612,-0.502752,10.0
4,-1.618309,-1.077198,-0.733954,-1.113964,-1.618114,-1.634060,-1.215218,0.559794,0.867215,-0.954412,-0.502752,5.0
5,-1.618306,-0.495465,-0.735008,-1.113964,-1.618114,-1.634060,-1.215218,0.980256,0.867215,-0.345818,-0.502752,10.0
6,-1.618299,-0.877446,-0.640567,0.975997,-1.618114,-1.634060,-1.215218,-0.521392,0.867215,-0.632215,-0.502752,18.0
7,-1.618246,-1.031640,-0.905019,-1.113964,-1.618114,-1.634060,-1.215218,0.019201,-1.309176,-1.002145,-0.502752,5.0
8,-1.618212,-1.031640,NaN,-1.113964,-1.618114,-1.634060,-1.215218,0.019201,-1.309176,0.012179,1.540349,90.0
9,-1.618205,-0.768809,-0.904204,-1.113964,-1.618114,-1.634060,-1.215218,-0.040865,0.867215,-0.512883,-0.502752,34.0


# Baseline model
Hier gaan we een baseline model maken met het gemiddelde en de mediaan

In [63]:
median = df[["stm_progfh_t_fh"]].median()

baseline = [median] * len(df)

print(f"R2 score: {r2_score(df['stm_progfh_t_fh'], baseline)}")

RMSE = np.sqrt(mean_squared_error(df['stm_progfh_t_fh'], baseline))
print(f"Root Mean Squared Error: {RMSE}")

baseline_info = {'baseline': baseline, 'rmse': RMSE}

with open('base_model.pkl', 'wb') as file:
    pickle.dump(baseline_info, file)

R2 score: -0.1125661828662543
Root Mean Squared Error: 75.98859706990203


## Conclusie
Met een score Baseline van -0.11372656130786285, kunnen we vastellen dat het gemiddelde een betere manier is om de stm_aann_t_fh te voorspellen dan de mediaan. Dit kunnen we zeggen omdat de score Baseline een negatief getal is. Een RMSE van 70.92134786646736 betekent dat de voorspelling er gemiddeld ~71 minuten naast de werkelijkheid zit.

# Model
Voor dit model gaan we gebruik maken van Random Forrest met bins. Eerst moeten we onze nominale waardes (zie [cleaned_features.ipynb](cleaned_features.ipynb)) omzetten met get_dummies naar bins.

In [64]:
def dummies_converter(df, value):
    dummies = pd.get_dummies(df[value])
    
    top_20_dummies = dummies.sum().nlargest(20).index
    
    df[value] = dummies[top_20_dummies].idxmax(axis=1)
    df.loc[~dummies[top_20_dummies].any(axis=1), value] = 9999
    
    dummies = pd.get_dummies(df[value], prefix='Index')
    dummies = dummies.astype(int)
    
    df = df.drop(columns=[value])
    df = pd.concat([df, dummies], axis=1)
    
    return df

for i in ['stm_geo_mld', 'stm_equipm_nr_mld', 'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst', 'stm_techn_gst']:
    df = dummies_converter(df, i)

Nu kunnen we het model maken met het gekozen algoritme

In [65]:
model = RandomForestRegressor(max_depth=10)

target = df["stm_progfh_t_fh"]
features = df.drop(columns=["stm_progfh_t_fh"])

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=10)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Square Error: {RMSE}")

model_info = {'model': model, 'rmse': RMSE}

with open('model.pkl', 'wb') as file:
    pickle.dump(model_info, file)

Root Mean Square Error: 45.614977923584746


In [55]:
# from sklearn.feature_selection import SelectFromModel

# # Eerst een Random Forest model trainen
# rf = RandomForestRegressor(n_estimators=100, max_depth=10)
# rf.fit(X_train, y_train)

# # Feature importance bekijken
# feature_importance = pd.DataFrame({'feature': X_train.columns, 'importance': rf.feature_importances_})
# feature_importance = feature_importance.sort_values('importance', ascending=False)
# print(feature_importance)

# # Selecteer features op basis van importance
# selector = SelectFromModel(rf, prefit=True)
# X_train_selected = selector.transform(X_train)
# X_test_selected = selector.transform(X_test)

# # Nieuwe feature namen
# selected_features = X_train.columns[selector.get_support()]

                        feature    importance
3            stm_progfh_in_duur  7.425935e-01
0              stm_sap_meld_ddt  7.599172e-02
2                stm_aanngeb_dd  4.718814e-02
125    Index_1.0507447424670093  1.366332e-02
1                stm_prioriteit  1.261791e-02
..                          ...           ...
61     Index_0.6835170353966592  2.686523e-08
115   Index_-1.6679005481942653  0.000000e+00
51    Index_-0.4324230834088187  0.000000e+00
121  Index_-0.03671337379750051  0.000000e+00
50   Index_-0.43494264787863646  0.000000e+00

[127 rows x 2 columns]


C:\Users\guusb\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\guusb\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:486: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


## Conclusie
Met een RMSE van 59.871659967304744 kunnen we zeggen dat het model nog niet goed genoeg is om te gebruiken om een voorspelling te maken